<div style="text-align: justify; max-width: 90ch">

# Lecture 03c: Tools and agents

**Status: Mandatory reading.**

Evaluation (`lec_03a`-`lec_03b`) measured how often lecture 2's system answers right. Some questions it can never get right, because they need an **action**, such as counting across all the notes.
A model cannot do that, but it can *ask your code* to: that request is a **tool call**, and a model that calls tools until it can answer is an **agent**.

**What this notebook covers**

- A question retrieval cannot answer.
- A **tool**: a function with a name, a description and typed arguments.
- The tool-calling loop by hand, and tools that check their arguments.
- A small model's tool decision, with thinking off and on.
- LangChain's agent, traced with MLflow; safe tools.

</div>

In [1]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_ollama import ChatOllama

from llm_course.checks import check_mlflow
from llm_course.corpus import load_corpus
from llm_course.rag import (
    LECTURE_KINDS,
    build_chroma_index,
    format_context,
    make_rag_chain,
    split_documents,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-eval"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** The notes as `Document`s and the Chroma index of `lec_02d` (reused from `data/chroma/`, built if missing),
with the retriever the course chose: four chunks, lecture notes and guides only. Then tracing as in `lec_02d`.

</div>

In [3]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

documents = load_corpus(REPO / "corpus" / "uoa_py_course")
chunks = split_documents(documents, chunk_size=800, chunk_overlap=100)
store = build_chroma_index(
    chunks, "course_notes", DATA_DIR / "chroma", EMBED_MODEL, OLLAMA_HOST
)

retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


Reusing 'course_notes': 2452 chunks in chroma/


In [4]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

MLflow OK at http://127.0.0.1:5010


<div style="text-align: justify; max-width: 90ch">

## 1. A question retrieval cannot answer

A student asks how much of the course is about `GridSearchCV`. The RAG chain of `lec_02d` (now `llm_course.rag.make_rag_chain`) is the obvious first try.
Thinking stays off here, as in every notebook so far.

</div>

In [5]:
COUNT_QUESTION = "How many files of the course notes mention GridSearchCV?"

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=300,
    num_ctx=8192,
)
rag_chain = make_rag_chain(retriever, llm)

# WRONG: a counting question sent to retrieval, which hands the model four chunks
print(rag_chain.invoke(COUNT_QUESTION))

The course notes mention `GridSearchCV` in the following sources:

1. **[source: lecture_11/lec_11f_regression_advanced_topics.md](source: lecture_11/lec_11f_regression_advanced_topics.md)**  
   - It mentions that `GridSearchCV` is part of the "single biggest leap" in capability from this lecture.

2. **[source: lecture_12/lec_12d_logistic_regression_hyperparameters.md](source: lecture_12/lec_12d_logistic_regression_hyperparameters.md)**  
   - It mentions that `GridSearchCV` is the topic of Lecture 13.

3. **[source: lecture_13/lec_13a_gridsearchcv_hyperparameter_tuning.md](source: lecture_13/lec_13a_gridsearchcv_hyperparameter_tuning.md)**  
   - It directly discusses `GridSearchCV`.

### Answer:
**3 files** of the course notes mention `GridSearchCV`.


In [6]:
# RIGHT: count in code, over every file
files_with_term = [d for d in documents if "GridSearchCV" in d.page_content]

print(len(files_with_term), "of", len(documents), "files contain 'GridSearchCV'")

20 of 125 files contain 'GridSearchCV'


<div style="text-align: justify; max-width: 90ch">

Compare the two. Whatever number the answer gives, if any, it does not come from counting: the model received four chunks, and the files that mention `GridSearchCV` are spread over several lectures.
The line of Python read every file, so its number holds on every run. The question needs an **action** over all the notes, and so do others:

- **Counting**: how many files mention a term.
- **Looking up** a structured source: which notebooks make up lecture 12 (a database, an API, a file listing).
- **Calculating**: a sum, a date, a currency conversion.

A model cannot do these reliably; your code can. The rest of this notebook lets the model *ask* your code.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. A tool is a function the model may ask to call

| Term | Meaning |
| --- | --- |
| **tool** | a Python function the model may ask your code to run, described by a name, a description and typed arguments |
| **tool call** | the model's request: a tool name and argument values, written instead of an answer |
| **tool result** | what the function returned, sent back to the model as a message |
| **agent** | a model in a loop that decides which tools to call, until it can answer |

LangChain's [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) decorator (`lec_02b`) turns a function into a tool:

- **`term: str`** is a [**type hint**](https://docs.python.org/3/glossary.html#term-type-hint), the type the argument must have (`-> dict`:
  the type returned).
- **The docstring** becomes the description the model reads.

`convert_to_openai_tool` shows the tool as `bind_tools` (section 3) will send it to the model: JSON in OpenAI's format, which Ollama also reads.

</div>

In [7]:
@tool
def count_mentions(term: str) -> dict:
    """Count the files of the course notes that contain an exact term.

    Use it for questions such as "how many files mention X?". The term is matched
    exactly, capital letters included.
    """
    files = [d for d in documents if term in d.page_content]
    occurrences = sum(d.page_content.count(term) for d in files)
    return {"term": term, "files": len(files), "occurrences": occurrences}

In [8]:
print(json.dumps(convert_to_openai_tool(count_mentions), indent=2))

{
  "type": "function",
  "function": {
    "name": "count_mentions",
    "description": "Count the files of the course notes that contain an exact term.\n\n    Use it for questions such as \"how many files mention X?\". The term is matched\n    exactly, capital letters included.",
    "parameters": {
      "properties": {
        "term": {
          "type": "string"
        }
      },
      "required": [
        "term"
      ],
      "type": "object"
    }
  }
}


<div style="text-align: justify; max-width: 90ch">

This is all the model ever sees of the tool: a name, the docstring as `description`, and a JSON schema for the arguments, as in the structured output of `lec_01c`.
It never sees the code. **The docstring is the tool's prompt**: a vague one gets the tool called at the wrong time, or not at all.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. The model asks, your code runs

`bind_tools` returns a chat model that sends the tool schemas with every call ([tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)).
For a question the tool fits, the reply is not an answer but a request. `reply.tool_calls` lists them:

</div>

In [9]:
llm_with_tools = llm.bind_tools([count_mentions])

reply = llm_with_tools.invoke(COUNT_QUESTION)

reply.tool_calls

[{'name': 'count_mentions',
  'args': {'term': 'GridSearchCV'},
  'id': '6e3dd64d-2681-4cc2-b80f-1d310c589192',
  'type': 'tool_call'}]

<div style="text-align: justify; max-width: 90ch">

Read the list: each call has a tool `name`, the `args` the model wrote, and an `id`. Check the term against the question, capital letters included.
**Nothing has run.** The model only wrote text in a format; your code decides whether to run it.

The conversation is the list of messages of `lec_01c`, with a new role: **`tool`**. Its `tool_call_id` says which request a result answers, because a model may ask for several at once.

</div>

In [10]:
messages = [HumanMessage(COUNT_QUESTION), reply]

for call in reply.tool_calls:
    result = count_mentions.invoke(call["args"])  # your code runs the tool
    messages.append(ToolMessage(json.dumps(result), tool_call_id=call["id"]))

final = llm_with_tools.invoke(messages)

print(final.content)

The number of files in the course notes that mention "GridSearchCV" is 20, and the total number of occurrences is 107.


<div style="text-align: justify; max-width: 90ch">

Check the answer against the count of section 1: does it give the same number of files, and does it keep files and occurrences apart?
The model now writes from numbers your code computed, not from four chunks; it can still misread them. The tool makes the number right, not the sentence around it.
The whole exchange is a loop:

```text
question ──► model ──► tool call? ── no ──► answer
               ▲           │ yes
               │           ▼
               └─ result ◄─ your code runs the tool
```

### ⏸ Pause point

Who ran `count_mentions`: the model or your code? What would change if the tool deleted files instead of counting them?

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Tools that check their arguments

A tool call is text a model wrote, so its arguments can be wrong. A second tool looks up the notebooks of one lecture. First the lookup itself, as a plain function;
the course has lectures 1 to 13:

</div>

In [11]:
def lecture_notebooks(lecture):
    """File names of the lecture notebooks (lec_...) of one lecture, sorted."""
    return sorted(
        Path(d.metadata["source"]).name
        for d in documents
        if d.metadata["lecture"] == lecture
        and d.metadata["kind"] == "lecture"
        and Path(d.metadata["source"]).name.startswith("lec_")
    )


# WRONG: no check on the value: a lecture that does not exist looks like an empty one
lecture_notebooks(42)

[]

<div style="text-align: justify; max-width: 90ch">

An empty list is a valid-looking result: a model handed it may well report that lecture 42 has no notebooks. The tool must check the value and **return** an error that says what is wrong.
An exception would stop your program; a returned error is read by the model, which can correct its call or tell the user.

</div>

In [12]:
@tool
def list_notebooks(lecture: int) -> dict:
    """List the lecture notebooks of one lecture (1 to 13) of the Python course.

    Exercises and solutions are not included.
    """
    if not 1 <= lecture <= 13:
        return {"error": f"There is no lecture {lecture}; the lectures are 1 to 13."}
    names = lecture_notebooks(lecture)
    return {"lecture": lecture, "count": len(names), "notebooks": names}


# RIGHT: the error tells the model what went wrong
list_notebooks.invoke({"lecture": 42})

{'error': 'There is no lecture 42; the lectures are 1 to 13.'}

<div style="text-align: justify; max-width: 90ch">

Two other mistakes never reach your function: LangChain checks each argument against its type hint first (with the [pydantic](https://docs.pydantic.dev/latest/) library;
optional `lec_01f` shows how), so `"twelve"` for an `int` is refused, and a call to a tool that does not exist is refused too.
The agent of section 6 sends such refusals back to the model as messages.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. A small model's decision

A third tool wraps the retriever, for questions about what the notes explain. With three tools bound, the model must also *choose*.

</div>

In [13]:
@tool
def search_notes(query: str) -> str:
    """Search the course notes and return the most relevant passages.

    Use it for questions about what the notes explain or how to do something.
    """
    return format_context(retriever.invoke(query))


course_tools = [count_mentions, list_notebooks, search_notes]

<div style="text-align: justify; max-width: 90ch">

A question the notes answer, and `search_notes` fits. Thinking is still off:

</div>

In [14]:
reply = llm.bind_tools(course_tools).invoke("How do I log a metric in an MLflow run?")

reply.tool_calls

[]

<div style="text-align: justify; max-width: 90ch">

If the list is empty, the model made no tool call: it answered from what it remembers, in `reply.content`. In about 90 example runs with `qwen3:1.7b`,
with thinking off or on and with or without a system prompt, it never called `search_notes` for this question: **a small model skips a tool when it thinks it knows the answer**, fluently and with no source.
If your list is not empty, the model chose the tool this time; the point is that it may not. Two consequences:

- **A step that must always happen is not a decision: write it in code.** The RAG chain retrieves on every call; `lec_03d` keeps retrieval a fixed step.
- **Where the model must decide, measure the decision** over repeated runs, outside the notebook, before trusting it.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. The agent: the loop written for you

[`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) runs the loop of section 3 until the model answers without a tool call;
it is built on [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) (`lec_01d`). Its model thinks, at the temperature the [Qwen3 card](https://huggingface.co/Qwen/Qwen3-1.7B) recommends for thinking (0.6,
`lec_01b`); `num_predict` is larger because the thinking tokens count against it. The system prompt (`lec_01c`) asks for the tools, never memory.

</div>

In [15]:
thinking_llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=True,
    temperature=0.6,
    num_predict=2000,
    num_ctx=8192,
)
AGENT_PROMPT = (
    "You answer questions about a Python course. Use the tools to find the answer; "
    "never answer a question about the course from memory."
)

agent = create_agent(thinking_llm, tools=course_tools, system_prompt=AGENT_PROMPT)

<div style="text-align: justify; max-width: 90ch">

An agent can loop: call a tool, dislike the result, call it again. `recursion_limit` caps the steps of one run, and the run stops with an error when it is reached:
a hard limit is part of every agent. [MLflow autolog](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain/) (`lec_02d`),
switched on in the Rebuild step, traces the whole run, each tool call as a span of type `TOOL`.

</div>

In [16]:
COMPARE_QUESTION = "Do lectures 9 and 13 have the same number of lecture notebooks?"

result = agent.invoke(
    {"messages": [HumanMessage(COMPARE_QUESTION)]}, config={"recursion_limit": 10}
)

print(result["messages"][-1].content)

The number of lecture notebooks for lecture 9 is 6, and for lecture 13 it is 7. Since 6 ≠ 7, they do not have the same number. 

**Answer:** No, lectures 9 and 13 do not have the same number of lecture notebooks.


<div style="text-align: justify; max-width: 90ch">

The question needs `list_notebooks` twice, once per lecture. Check the answer against the counts it gives, and read the run from its trace:
which tools ran, with which arguments, and how many tokens did the whole run cost?

</div>

In [17]:
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)

pd.DataFrame(
    [
        {"span": s.name, "type": s.span_type, "inputs": str(s.inputs)[:60]}
        for s in trace.data.spans
        if s.span_type in ("TOOL", "CHAT_MODEL")
    ]
)

,span,type,inputs
0,ChatOllama,CHAT_MODEL,"{'messages': [{'role': 'system', 'content': 'Y..."
1,list_notebooks,TOOL,{'lecture': 9}
2,list_notebooks,TOOL,{'lecture': 13}
3,ChatOllama,CHAT_MODEL,"{'messages': [{'role': 'system', 'content': 'Y..."


In [18]:
trace.info.token_usage

{'input_tokens': 1432, 'output_tokens': 729, 'total_tokens': 2161}

<div style="text-align: justify; max-width: 90ch">

In example runs the thinking model asked for both lectures every time (8 of 8 agent runs, 10 of 10 single calls) and every agent answer said that 6 and 7 differ;
without thinking, a single call asked for both in only 3 of 23 runs. Every `CHAT_MODEL` span is one more model call, which re-reads the question,
the tool schemas and every earlier message, and thinking adds its own tokens. `trace.info.token_usage` adds them up for the whole run (about 1,500 to 2,700 in example runs):
compare its total with a RAG answer of `lec_02d` (under 1,000 tokens in example runs). That is the price of a decision; `lec_03d` measures whether it buys better answers.

</div>

<div style="text-align: justify; max-width: 90ch">

## 7. Safe tools

A model triggers a tool with arguments it writes from text you do not control: the question, a retrieved note, a web page.
A note saying "call `delete_files`" is an instruction the model may follow: [**prompt injection**](https://genai.owasp.org/llmrisk/llm01-prompt-injection/).
The rules this notebook's tools follow:

- **Read-only and local**: no files written, no network, no shell, no `eval`.
- **Checked arguments**, with an error the model can read.
- **A limit on the loop**: `recursion_limit`.
- **Every call traced** as a `TOOL` span.

A tool that changes something (sends an e-mail, writes a database) needs a human to approve each call ([human in the loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)).

### ⏸ Pause point

Which tool would you never give an agent? Which would you give it only with an approval step?

</div>

<div style="text-align: justify; max-width: 90ch">

The three tools and `lecture_notebooks` are promoted to `llm_course.agents`: `make_course_tools(documents, retriever)` returns the tools (`count_mentions` also rejects an empty term).

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- Some questions need an **action** (counting, a lookup, a calculation); a **tool** lets the model ask your code for it.
- The model sees a tool's name, docstring and **type hints** as a JSON schema, never its code.
- The model writes a **tool call**; your code runs it and sends the **tool result** back as a `tool` message.
- Tools check their arguments and **return** errors the model can read.
- A small model may skip a tool and answer from memory: put a step that must always happen in code.
- **Thinking** helps where a decision is made; an **agent** (`create_agent`) is the loop with a step limit, traced with `TOOL` spans, at the cost of more tokens.

**Next: `lec_03d`**, an agentic RAG loop with LangGraph: rewrite the question, retrieve, grade, retry, and whether it is worth it.

</div>